**Исходные данные**

**oбъем данных:** 25,264 записи об отключениях

**Период:** Различные временные метки (требовали преобразования)

**Источники:** База данных SQLite с таблицами blackouts, buildings, blackouts_buildings

**Выполненные этапы очистки**
**1. Преобразование временных меток**
Проблема: Даты в строковом формате
Решение: Конвертация в datetime с безопасной обработкой ошибок
Результат: 25,264 корректных временных меток

**2. Расчет ключевых метрик**
Длительность отключений в часах и днях

Фильтрация аномалий: Удалены записи с отрицательной длительностью

Устранение дубликатов: Обеспечена уникальность записей

**3. Обогащение данных**
Добавлено: Количество затронутых зданий для каждого отключения
Метод: LEFT JOIN с таблицей связей blackouts_buildings
Результат: Метрика масштаба воздействия для каждого инцидента

**4. Стандартизация текстовых данных**
Обработаны поля:

type - тип отключения

source - источник информации

initiator_name - инициатор отключения

Достигнуто: Единый формат (нижний регистр, без пробелов)

**Критическая проблема и решение**

Проблема
65% пропусков в поле source (16,542 записи из 25,264)

Анализ причин

Исследование показало, что значения initiator_name могут служить надежным суррогатом для источника, когда он не указан.

Решение
```python
blackouts['source'] = blackouts['source'].fillna(blackouts['initiator_name'])
```

In [27]:
import sqlite3
import pandas as pd

# Подключаемся к базе данных
conn = sqlite3.connect(r"C:\Users\igor\Рабочий стол\Кейс_Аналитика.db")

# таблицы в базе данных
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

                  name
0               cities
1            districts
2       folk_districts
3   big_folk_districts
4              streets
5            buildings
6            blackouts
7  blackouts_buildings


In [28]:
blackouts = pd.read_sql("SELECT * FROM blackouts;", conn)
blackouts.head()

,id,start_date,end_date,description,type,initiator_name,source
0,f88cefa506f44ebf8f010b8681b5449e,2018-01-01 00:08:00,2018-01-01 09:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
1,38ddf6852801fa90cc70f9770239961e,2018-01-01 00:24:00,2018-01-01 10:00:00,"Авария на электролинии, остановка работы насос...",cold_water,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
2,53c570099fe380dce9e56e2ace9cfa9c,2018-01-01 10:44:00,2018-01-02 18:00:00,Авария в системе водоснабжения дома. Жителям н...,hot_water,"ООО ""Управляющая компания ""Регион-ЖКХ""","Аварийная служба ООО ""Мадикс"""
3,9c1b9ebbd9a698eef046b27cb3568745,2018-01-01 11:32:00,2018-01-01 15:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...
4,8aa631cb343aac0731bbde806dfa6d8c,2018-01-01 11:33:00,2018-01-01 15:00:00,"Авария на электролинии, остановка работы насос...",hot_water,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...


In [29]:
blackouts.info()
blackouts.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25264 entries, 0 to 25263
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              25264 non-null  object
 1   start_date      25264 non-null  object
 2   end_date        25264 non-null  object
 3   description     25264 non-null  object
 4   type            25264 non-null  object
 5   initiator_name  25264 non-null  object
 6   source          8722 non-null   object
dtypes: object(7)
memory usage: 1.3+ MB


id                    0
start_date            0
end_date              0
description           0
type                  0
initiator_name        0
source            16542
dtype: int64

In [30]:
blackouts['start_date'] = pd.to_datetime(blackouts['start_date'], errors='coerce')
blackouts['end_date'] = pd.to_datetime(blackouts['end_date'], errors='coerce')

In [31]:
blackouts['has_source'] = blackouts['source'].notna().astype(int)

In [32]:
blackouts['duration_hours'] = (blackouts['end_date'] - blackouts['start_date']).dt.total_seconds() / 3600
blackouts['duration_days'] = blackouts['duration_hours'] / 24

In [33]:
blackouts = blackouts[blackouts['duration_hours'] >= 0]
blackouts = blackouts.drop_duplicates(subset='id')

In [34]:
# выбираем строки, где source == 'неизвестен'
mask = blackouts['source'].isna()

# смотрим, какие значения initiator_name у этих строк
blackouts.loc[mask, 'initiator_name'].value_counts()

initiator_name
МУПВ ВПЭС (электрические сети)              1650
КГУП «Приморский водоканал»                  859
ООО "Мингородок"                             704
ООО "Невельского"                            589
ООО "Радон"                                  435
                                            ... 
ТСЖ "На Кирова, 25Б"                           1
ТСЖ "Некрасовская-92"                          1
ЖСК № 77                                       1
ТСЖ "ДАВЫДОВА, 42 А"                           1
Аварийная служба ООО "УК "Снеговая падь"       1
Name: count, Length: 336, dtype: int64

In [35]:
blackouts['source'] = blackouts['source'].fillna(blackouts['initiator_name'])

In [38]:
blackouts_buildings = pd.read_sql("SELECT * FROM blackouts_buildings;", conn)

# Считаем, сколько зданий связано с каждым отключением
affected_counts = blackouts_buildings.groupby('blackout_id').size().reset_index(name='affected_buildings_count')

# Объединяем с основной таблицей
blackouts = blackouts.merge(affected_counts, how='left', left_on='id', right_on='blackout_id')
blackouts.drop(columns=['blackout_id'], inplace=True)
blackouts['affected_buildings_count'] = blackouts['affected_buildings_count'].fillna(0).astype(int)

In [40]:
for col in ['type', 'source', 'initiator_name']:
    blackouts[col] = blackouts[col].astype(str).str.strip().str.lower()

In [54]:
blackouts.to_csv('cleaned_blackouts.csv', index=False, encoding='utf-8')